<a href="https://colab.research.google.com/github/HarshulAgarwal07/digitsofPi_cuda/blob/main/PI_Digits.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
!pip install nvcc4jupyter
%load_ext nvcc4jupyter

The nvcc4jupyter extension is already loaded. To reload it, use:
  %reload_ext nvcc4jupyter


In [14]:
# Detect selected GPU and its NVIDA architecture:
import subprocess
gpu_info = subprocess.getoutput("nvidia-smi --query-gpu=name,compute_cap --format=csv,noheader,nounits")
if "not found" in gpu_info.lower(): raise RuntimeError("Error: No GPU found. Please select a GPU runtime environment.")
gpu_name, compute_cap = map(str.strip, gpu_info.split(','))
gpu_arch = f"sm_{compute_cap.replace('.', '')}"

print(f"{'GPU Name':<15}: {gpu_name}")
print(f"{'Architecture':<15}: {gpu_arch}")

GPU Name       : Tesla T4
Architecture   : sm_75


Initiialize the

*   Block size
*   Number of iterations to perform per thread
*   Number of blocks

The function performs parallel sum reduction for a single warp using the shfl_down_sync function which directly accesses registers instead of shared memory.




In [15]:
%%writefile helpers.cuh
#include <iostream>
#include <curand_kernel.h>
#include <time.h>
#define ll unsigned long long
using namespace std;

const int BLOCK_SIZE = 512; //threads per block
const int ITERATIONS = 1e5;
const int NBLOCKS = 256;


__device__ ll warp_reduce_sum(ll val) {
    for(int offset = warpSize/2; offset > 0; offset /= 2) {
        val += __shfl_down_sync(0xffffffff, val, offset); // 0xffffffff means all 32 threads are active
    }
    return val;
}

Overwriting helpers.cuh


The picount shared function uses Monte Carlo simulation to generate the digits of pi

In [20]:

%%writefile kernel.cuh
#include "helpers.cuh"

__global__ void picount_shared(ll* dTotal) {

    int global_tid = blockIdx.x * blockDim.x + threadIdx.x;

    int local_tid = threadIdx.x;  //0-BLOCK_SIZE-1 threads

    __shared__ ll cache[BLOCK_SIZE];

    //Initialize RNG
    curandState_t rng;
    curand_init(clock64(), global_tid, 0, &rng);


    //Monte Carlo Simulation
    ll local_count=0;
    for(int i=0; i<ITERATIONS; i++) {
        float x = curand_uniform(&rng);
        float y = curand_uniform(&rng);
        if(x*x + y*y < 1)
          local_count++;
    }

    //Load local count into shared memory
    cache[local_tid] = local_count;
    __syncthreads();

    // Reduction Phase 1: Shared Memory
    int i = blockDim.x/2;
    while(i > 32) {
        if(local_tid < i) {
            cache[local_tid] += cache[local_tid + i];
        }
        __syncthreads();
        i /= 2;
    }

    // Reduction Phase 2: Warp Shuffle
    if (local_tid < 32) {
        ll warp_sum = cache[local_tid];

        // Add the upper half of the last shared memory fold
        if (blockDim.x > 32) {
             warp_sum += cache[local_tid + 32];
        }

        // Reduce the warp
        warp_sum = warp_reduce_sum(warp_sum);

        // Final Atomic Add
        if (local_tid == 0) {
            atomicAdd(dTotal, warp_sum);
        }
    }
}



Overwriting kernel.cuh


The main function launches the kernel and measures the time taken to find the estimated pi.

In [18]:
%%writefile main.cu
#include <iostream>
#include <time.h>
#include "kernel.cuh"
using namespace std;

int main() {
    ll* dTotal;
    ll hTotal;

    cudaMalloc(&dTotal, sizeof(ll));
    cudaMemset(dTotal, 0, sizeof(ll));

    // Time the kernel
    struct timespec start, stop;
    clock_gettime(CLOCK_REALTIME, &start);

    // Launch the kernel
    picount_shared<<<NBLOCKS, BLOCK_SIZE>>>(dTotal);
    cudaDeviceSynchronize();

    clock_gettime(CLOCK_REALTIME, &stop);

    cudaMemcpy(&hTotal, dTotal, sizeof(ll), cudaMemcpyDeviceToHost);


    ll tests = (ll)NBLOCKS*ITERATIONS*BLOCK_SIZE;

    cout << "Approximated PI using " << tests << " random tests\n";
    double pi = 3.1415926535897932384626433;

    double estimated_pi = 4.0 * (double)hTotal / (double)tests;
    cout.precision(15);
    cout << "Estimated Pi: " << estimated_pi << endl;
    cout << "Error: " << abs(estimated_pi - pi) << endl;

    double accum = (stop.tv_sec - start.tv_sec) + (double)(stop.tv_nsec - start.tv_nsec) / 1.0e9;
    cout << "Time: " << accum << " seconds" << endl;

    cudaFree(dTotal);
    return 0;
}

Writing main.cu


Monte

In [21]:
!nvcc -arch=sm_75 -o monte_carlo main.cu && ./monte_carlo

kernel.cuh(11): error: identifier "global_tid" is undefined
      curand_init(clock64(), global_tid, 0, &rng);
                             ^

1 error detected in the compilation of "main.cu".
